In [20]:
import pandas as pd
import os
import random
from dotenv import load_dotenv
from openai import OpenAI
from utils.token_usage import (
    create_usage_stats,
    update_usage,
    save_token_usage_log,
)
from config.constants import (
    CONTEXT_REL_CONFIG,
    SLEEP_BETWEEN_CALLS,
    DATA_PATH,
)

load_dotenv(override=True)

model_config = CONTEXT_REL_CONFIG

MODEL_NAME = model_config["name"]
REASONING_EFFORT = model_config["reasoning_effort"]

print(f"Using model: {MODEL_NAME} with reasoning effort: {REASONING_EFFORT}")


OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)

client = OpenAI(
    api_key=OPENAI_API_KEY
)

# df=pd.read_csv(f'{DATA_PATH}df_n1.csv')


usage_stats = create_usage_stats()

Using model: gpt-5.6-luna with reasoning effort: low


In [21]:
df=df_new.copy()

In [22]:


def build_context_messages(row):

    system_prompt = """
You are an expert software engineer performing context compression for an automated code review system.

Your task is NOT to summarize the file and NOT to reproduce the original file.
Your task is to extract the MINIMUM amount of source code required for another LLM to correctly review the given code hunk.

You will receive:
1. The original source file (<old_file>)
2. The changed code hunk (<hunk>)

Your goal:
Produce a compact code context that preserves only the information necessary to understand the behavior, dependencies, and potential issues of <hunk>.

Selection strategy:
- Start by including only the changed hunk.
- Add the smallest enclosing scope needed (function, method, or class).
- Add external definitions only when the hunk depends on them and their absence would make the behavior ambiguous.
- Add called functions, variables, attributes, imports, decorators, or parent classes only when they directly influence the logic of the hunk.
- Prefer short relevant snippets over complete files.
- If a definition is large, include only the relevant parts.

Strict exclusion rules:
- Do NOT return the entire file if it's loo large.
- Do NOT include file headers, licenses, comments, documentation, or unrelated code.
- Do NOT include neighboring functions unless they are required to understand the hunk.
- Do NOT include imports unless they are necessary to understand a referenced component.
- Do NOT include boilerplate code.

Think like a human reviewer:
A reviewer does not read the whole repository file. They inspect the changed code and only open the definitions required to reason about correctness.

Output requirements:
- Return ONLY the extracted source code.
- No explanations.
- No markdown fences.
- No comments about your selection process.

The final output should be significantly smaller than <old_file>.
"""

    user_prompt = f"""
<old_file>
{row["oldf"]}
</old_file>


<hunk>
{row["hunk"]}
</hunk>
"""

    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]


# ==========================
# Output parsing
# ==========================

def parse_context_output(text):

    if text is None:
        return ""

    return text.strip()


# ==========================
# OpenAI prediction
# ==========================

def predict_context(row):

    messages = build_context_messages(row)

    kwargs = {
        "model": MODEL_NAME,
        "messages": messages,
        "timeout": 120,
    }

    if REASONING_EFFORT:
        kwargs["reasoning_effort"] = REASONING_EFFORT

    response = client.chat.completions.create(**kwargs)

    # update_usage(response.usage)
    update_usage(
        usage_stats,
        response.usage
    )

    raw_text = (
        response
        .choices[0]
        .message
        .content
    )

    context = parse_context_output(
        raw_text
    )

    return {
        "relevant_context": context
    }


# ==========================
# Pipeline execution
# ==========================

def run_context_relevance_pipeline(df):

    preds = []

    total = len(df)

    processed = 0

    base_sleep = SLEEP_BETWEEN_CALLS

    print(
        f"[START] Processing {total} rows"
    )

    for _, row in df.iterrows():

        processed += 1

        print(
            f"\n[ROW {processed}/{total}] Starting"
        )

        if (
            pd.isna(row["hunk"])
            or pd.isna(row["oldf"])
        ):

            print(
                "[SKIP] Missing old file or hunk"
            )

            preds.append(
                {
                    "relevant_context": ""
                }
            )

            continue

        success = False

        sleep_time = base_sleep

        for attempt in range(3):

            try:

                pred = predict_context(row)

                preds.append(
                    pred
                )

                success = True

                break

            except Exception as e:

                print(
                    f"[ERROR] {e}"
                )

                wait = (
                    sleep_time
                    + random.uniform(0, 1)
                )

                print(
                    f"[RETRY] waiting {wait:.2f}s"
                )

                time.sleep(wait)

                sleep_time *= 2

        if not success:

            preds.append(
                {
                    "relevant_context": ""
                }
            )

        wait = (
            base_sleep
            + random.uniform(0, 0.8)
        )

        time.sleep(wait)

        if processed % 10 == 0:

            print(
                f"[CHECKPOINT] processed {processed}/{total}"
            )

    print(
        "\n[DONE] Building dataframe"
    )

    pred_df = pd.DataFrame(
        preds
    )

    df = df.reset_index(
        drop=True
    )

    return pd.concat(
        [
            df,
            pred_df
        ],
        axis=1
    )

In [26]:
import time
import tiktoken


start_time = time.perf_counter()

df_result = run_context_relevance_pipeline(df)

execution_duration = time.perf_counter() - start_time
df_relevant_context = df_result[
    ["patch_id", "relevant_context"]
]

df_relevant_context.to_csv(
    f"{DATA_PATH}df_n2_1_new.csv",
    index=False
)




[START] Processing 417 rows

[ROW 1/417] Starting

[ROW 2/417] Starting

[ROW 3/417] Starting

[ROW 4/417] Starting

[ROW 5/417] Starting

[ROW 6/417] Starting

[ROW 7/417] Starting

[ROW 8/417] Starting

[ROW 9/417] Starting

[ROW 10/417] Starting
[CHECKPOINT] processed 10/417

[ROW 11/417] Starting

[ROW 12/417] Starting

[ROW 13/417] Starting

[ROW 14/417] Starting

[ROW 15/417] Starting

[ROW 16/417] Starting

[ROW 17/417] Starting

[ROW 18/417] Starting

[ROW 19/417] Starting

[ROW 20/417] Starting
[CHECKPOINT] processed 20/417

[ROW 21/417] Starting

[ROW 22/417] Starting

[ROW 23/417] Starting

[ROW 24/417] Starting

[ROW 25/417] Starting

[ROW 26/417] Starting

[ROW 27/417] Starting

[ROW 28/417] Starting

[ROW 29/417] Starting

[ROW 30/417] Starting
[CHECKPOINT] processed 30/417

[ROW 31/417] Starting

[ROW 32/417] Starting

[ROW 33/417] Starting

[ROW 34/417] Starting

[ROW 35/417] Starting

[ROW 36/417] Starting

[ROW 37/417] Starting

[ROW 38/417] Starting

[ROW 39/417] Sta

In [31]:
df_n1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 505 entries, 0 to 504
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   old_hunk                  505 non-null    object
 1   oldf                      505 non-null    object
 2   hunk                      505 non-null    object
 3   comment                   505 non-null    object
 4   ids                       505 non-null    object
 5   repo                      505 non-null    object
 6   ghid                      505 non-null    int64 
 7   old                       505 non-null    object
 8   new                       505 non-null    object
 9   lang                      505 non-null    object
 10  pr_number                 505 non-null    int64 
 11  pr_title                  505 non-null    object
 12  changed_files             505 non-null    object
 13  target_file               505 non-null    object
 14  num_code_hunks            

In [36]:
df_n1=pd.read_csv('../data/df_n1.csv')

In [ ]:
df_n1['relevant_context  ']=df_relevant_context['relevant_context'] when df_n1['patch_id']==df_relevant_context['patch_id']

In [30]:
df_n1 = df_n1.merge(
    df_relevant_context[["patch_id", "relevant_context"]],
    on="patch_id",
    how="left"
)

In [32]:
df_old.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 22 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   old_hunk                             100 non-null    object
 1   oldf                                 100 non-null    object
 2   hunk                                 100 non-null    object
 3   comment                              100 non-null    object
 4   ids                                  100 non-null    object
 5   repo                                 100 non-null    object
 6   ghid                                 100 non-null    int64 
 7   old                                  100 non-null    object
 8   new                                  100 non-null    object
 9   lang                                 100 non-null    object
 10  patch_id                             100 non-null    object
 11  pr_number                            100 non-n

In [37]:
# 1. Remplir relevant_context avec df_relevant_context via patch_id
context_1 = df_relevant_context.set_index("patch_id")["relevant_context"]

df_n1["relevant_context"] = df_n1["patch_id"].map(context_1)


# 2. Compléter les valeurs manquantes avec df_old via ids
context_2 = df_old.set_index("ids")["relevant_context"]

df_n1["relevant_context"] = (
    df_n1["relevant_context"]
    .fillna(df_n1["ids"].map(context_2))
)

In [39]:
df_n1[['patch_id','relevant_context']].to_csv('../data/df_n2_1.csv',index=False)

In [8]:


# ==========================
# Token counting
# ==========================

encoding = tiktoken.get_encoding("o200k_base")


def count_tokens(text):
    if text is None:
        return 0

    return len(encoding.encode(str(text)))


# ==========================
# Compute compression statistics
# ==========================

total_old_tokens = (
    df_result["oldf"]
    .apply(count_tokens)
    .sum()
)

total_generated_tokens = (
    df_result["relevant_context"]
    .apply(count_tokens)
    .sum()
)

total_reduced_tokens = (
    total_old_tokens - total_generated_tokens
)

compression_ratio = (
    total_generated_tokens / total_old_tokens
    if total_old_tokens
    else 0
)

reduction_ratio = 1 - compression_ratio

compression_factor = (
    total_old_tokens / total_generated_tokens
    if total_generated_tokens
    else 0
)


# ==========================
# Context statistics
# ==========================

context_statistics = {
    "dataset_length": len(df_result),

    "context_statistics": {
        "total_original_context_tokens": int(
            total_old_tokens
        ),

        "total_extracted_context_tokens": int(
            total_generated_tokens
        ),

        "total_removed_tokens": int(
            total_reduced_tokens
        ),

        "context_reduction_percentage": round(
            reduction_ratio * 100,
            2
        ),

        "compression_factor": round(
            compression_factor,
            2
        ),
    }
}


save_token_usage_log(
    usage_stats=usage_stats,
    model_name=model_config["name"],
    task_name="in_file_context_compression",
    execution_duration=execution_duration,
    additional_stats=context_statistics,
)




Token usage saved to ..\logs\token_usage_insights_logs.json


In [6]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 50)

In [13]:
import pandas as pd
df_old=pd.read_csv('../data/old/df_n2.csv')

In [12]:
df["comment"].is_unique

True

In [18]:
df_old.head()

,old_hunk,oldf,hunk,comment,ids,repo,ghid,old,new,lang,patch_id,pr_number,pr_title,changed_files,target_file,num_code_hunks,pr_code_hunks,same_file_code_hunks,code_hunks_num_same_file,relevant_context,relevant_different_files_code_hunks,relevant_same_file_code_hunks
0,"@@ -41,7 +41,6 @@ class CachedObject(namedtupl...","""""""\nCaching utilities for zipline\n""""""\nfrom ...","@@ -41,6 +41,7 @@ class CachedObject(namedtupl...","We still need this line, this is what is causi...","[16998, '9b8bb1d7ffd47d868b78f25cefe3ed6fd032a...",quantopian/zipline,1319,>>> obj.unwrap(expires)\n 1\n >>>...,>>> obj.unwrap(expires)\n 1\n >>>...,py,P000050,1319,NamedTemp manual deletion for windows users.,['zipline/utils/cache.py'],zipline/utils/cache.py,1,"[{'filename': 'zipline/utils/cache.py', 'patch...","[{'filename': 'zipline/utils/cache.py', 'patch...",4,"class CachedObject(namedtuple(""_CachedObject"",...",[],"[{'filename': 'zipline/utils/cache.py', 'patch..."
1,"@@ -5,6 +5,9 @@\n \n from __future__ import wi...","# -*- coding: utf-8 -*-\n""""""A Fabric fabfile w...","@@ -6,7 +6,7 @@ BigchainDB, including its stor...",you can just do: ``` python import sys ``` and...,"[9769, '87275b966ce2003bf97523ddac0a8bc66a1a62...",bigchaindb/bigchaindb,316,"from __future__ import with_statement, unicod...","from __future__ import with_statement, unicod...",py,P000051,316,Better Support for New Relic Server Monitoring,"['deploy-cluster-aws/fabfile.py', 'docs/source...",deploy-cluster-aws/fabfile.py,2,"[{'filename': 'deploy-cluster-aws/fabfile.py',...","[{'filename': 'deploy-cluster-aws/fabfile.py',...",3,from os import environ\n\nimport sys\n\n@task\...,"[{'filename': 'docs/source/deploy-on-aws.md', ...","[{'filename': 'deploy-cluster-aws/fabfile.py',..."
2,"@@ -550,13 +550,20 @@ def contains_nested_data...","# Copyright (c) 2017-2021, NVIDIA CORPORATION ...","@@ -552,10 +552,10 @@ Parameters\n \n def ...",```suggestion # The sole point of this call is...,"[54174, '0cc47219dd85ca33c730b8af52a604bea2b6d...",NVIDIA/DALI,3245,def _setup_pipe_pool_dependency(self):\n ...,def _setup_pipe_pool_dependency(self):\n ...,py,P000052,3245,Ensure keeping py_pool alive until pipline is ...,"['dali/python/backend_impl.cc', 'dali/python/n...",dali/python/nvidia/dali/pipeline.py,5,"[{'filename': 'dali/python/backend_impl.cc', '...",[{'filename': 'dali/python/nvidia/dali/pipelin...,2,def _setup_pipe_pool_dependency(self):\n ...,"[{'filename': 'dali/python/backend_impl.cc', '...",[{'filename': 'dali/python/nvidia/dali/pipelin...
3,"@@ -60,18 +66,20 @@ def __init__(self, mode='t...",""""""" PPIDataset for inductive learning. """"""\nim...","@@ -75,7 +75,7 @@ class PPIDataset(DGLBuiltinD...","if `graph` is a property, why not keep using it?","[38808, 'f285b2a20a449dd3feb634b970abba527d03a...",dmlc/dgl,1804,g_data = json.load(open(graph_file))\...,g_data = json.load(open(graph_file))\...,py,P000053,1804,[Dataset] PPIDataset,"['examples/pytorch/cluster_gcn/README.md', 'ex...",python/dgl/data/ppi.py,6,[{'filename': 'examples/pytorch/cluster_gcn/RE...,"[{'filename': 'python/dgl/data/ppi.py', 'patch...",4,import json\nimport numpy as np\nimport networ...,[{'filename': 'examples/pytorch/gat/train_ppi....,"[{'filename': 'python/dgl/data/ppi.py', 'patch..."
4,"@@ -62,7 +62,7 @@ def update_web_location(self...",# -*- coding: utf-8 -*-\n\nimport datetime\nim...,"@@ -62,7 +62,7 @@ class PokemonGoBot(object):\...",It's unfortunate that you have so many changes...,"[21792, 'a710a6e102ebbb44820d9546fcc3ccc20ae15...",PokemonGoF/PokemonGo-Bot,939,status = map_objects.get('status'...,status = map_objects.get('status'...,py,P000054,939,"SpiralNavigator rewrite, mode switch","['pokemongo_bot/__init__.py', 'pokemongo_bot/s...",pokemongo_bot/__init__.py,2,"[{'filename': 'pokemongo_bot/__init__.py', 'pa...","[{'filename': 'pokemongo_bot/__init__.py', 'pa...",1,class PokemonGoBot(object):\n def update_we...,[],[]


In [15]:
df=pd.read_csv('../data/df_n1.csv')

In [16]:
df_new = df[
    ~df["comment"].isin(df_old["comment"])
]

In [ ]:
df_new = df[
    ~df["comment"].isin(df_old["comment"])
]

In [ ]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 417 entries, 1 to 504
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   old_hunk                  417 non-null    object
 1   oldf                      417 non-null    object
 2   hunk                      417 non-null    object
 3   comment                   417 non-null    object
 4   ids                       417 non-null    object
 5   repo                      417 non-null    object
 6   ghid                      417 non-null    int64 
 7   old                       417 non-null    object
 8   new                       417 non-null    object
 9   lang                      417 non-null    object
 10  pr_number                 417 non-null    int64 
 11  pr_title                  417 non-null    object
 12  changed_files             417 non-null    object
 13  target_file               417 non-null    object
 14  num_code_hunks            417 n